<a href="https://colab.research.google.com/github/ZeroFiles/AML-Final-Ortiz-Larry/blob/main/Tratamiento_de_Datos_Sinteticos_DataSet_CTGAN_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/ZeroFiles/AML-Final-Ortiz-Larry/blob/main/notebooks/Tratamiento_de_Datos_Sinteticos_DataSet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tratamiento de datos + generación sintética controlada con CTGAN

Este notebook prepara una base agregada por local-hora y, cuando el histórico real es insuficiente, permite generar datos sintéticos condicionados para cubrir combinaciones temporales faltantes.

Criterio metodológico:
- Los registros reales se conservan como fuente principal.
- Los registros sintéticos se marcan con `origen_dato = "sintetico_ctgan"`.
- Las variables temporales dependientes de continuidad, como lags y rolling, se recalculan después de unir real + sintético.
- La variable `flota_requerida_estimada` no se incluye como feature para evitar fuga de información.


In [1]:
# =====================================================
# 00) MONTAJE DE GOOGLE DRIVE
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/AML_Final_Project/'

print("✅ Google Drive montado")
print(f"📂 Ruta base del proyecto: {BASE_PATH}")


Mounted at /content/drive
✅ Google Drive montado
📂 Ruta base del proyecto: /content/drive/MyDrive/AML_Final_Project/


In [2]:
# =====================================================
# 01) LIBRERÍAS
# =====================================================

# En Colab, descomentar si SDV no está instalado
!pip install -q sdv pyarrow openpyxl

import pandas as pd
import numpy as np

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from sdv.sampling import DataFrameCondition
from sdv.evaluation.single_table import evaluate_quality

pd.set_option("display.max_columns", 100)

print("✅ Librerías cargadas")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.8/204.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.3 MB/s eta 0:00:00
✅ Librerías cargadas


In [3]:
# =====================================================
# 02) PARÁMETROS
# =====================================================

INPUT_FILE = BASE_PATH + "datasetPrevio.xlsx"

OUTPUT_FILE_REAL = BASE_PATH + "dataset_model_real.parquet"
OUTPUT_FILE_SYNTH = BASE_PATH + "dataset_sintetico_ctgan.parquet"
OUTPUT_FILE_AUGMENTED = BASE_PATH + "dataset_model_real_mas_sintetico.parquet"

# Si quieres alinear horas a Perú
USE_LIMA_TZ = True
LIMA_TZ = "America/Lima"

# Frecuencia horaria
FREQ = "h"

# Filtrado por horas operativas detectadas por local
FILTER_OPERATING_HOURS = True
OPER_HOUR_MIN_POS_RATE = 0.02

# Generación sintética
USE_SYNTHETIC_DATA = True
CTGAN_EPOCHS = 300

# Para ejecución rápida de prueba, limitar las filas faltantes a sintetizar.
# Usar None para generar todas las combinaciones faltantes.
MAX_SYNTHETIC_ROWS = None

# Lags / rolling
LAGS = [1, 2, 3, 24, 168]
ROLL_WINDOWS = [6, 12, 24, 168]

# Columnas base que CTGAN aprenderá.
# No incluir lags, rolling, ts_hour ni flota_requerida_estimada.
CTGAN_COLUMNS = [
    "local",
    "pedidos",
    "km_mean",
    "t_ret_mean",
    "t_ret_p75",
    "hora",
    "dow",
    "month",
    "is_weekend"
]

CONDITION_COLUMNS = [
    "local",
    "hora",
    "dow",
    "month",
    "is_weekend"
]

print("✅ Parámetros configurados")


✅ Parámetros configurados


In [4]:
# =====================================================
# 03) CARGA
# =====================================================

df = pd.read_excel(INPUT_FILE)

print("Shape original:", df.shape)
display(df.head())


Shape original: (9504, 11)


,order_id,organizacion,local,id_conductor,fecha_creacion_fv,KM_REAL,TIEMPO_ENTREGA_RETORNO,hora_inicial,fecha_creacion_date,real_tiempo_inicial,real_tiempo_final
0,68184850085a6b43df0b4825,El Tablón,El Tablón - Mariscal,67e480b0db09aaf63829b783,2025-05-04 23:52:18.750,2.247111,80.0,23,2025-05-04,2025-05-05 00:18:35 UTC,2025-05-05 01:09:13 UTC
1,68184641085a6b43df0b481e,El Tablón,El Tablón - Mariscal,67e480b0db09aaf63829b783,2025-05-04 23:38:17.595,1.295928,40.0,23,2025-05-04,2025-05-05 00:18:35 UTC,2025-05-05 00:52:41 UTC
2,67b0157d87d7b6fdd1bb2a27,El Tablón,El Tablón - Alameda,675cc31658a3f0c5eff208d0,2025-02-14 23:13:23.090,0.001111,2.0,23,2025-02-14,2025-02-15 00:10:28 UTC,2025-02-15 00:11:28 UTC
3,67af7fb987d7b6fdd1bb1048,El Tablón,El Tablón - Alameda,676b165798b6303093957e58,2025-02-14 12:34:02.457,0.042991,122.0,12,2025-02-14,2025-02-14 12:41:28 UTC,2025-02-14 13:42:08 UTC
4,67af65ba87d7b6fdd1bb086d,El Tablón,El Tablón - Mariscal,677335ba98b630309395d2c8,2025-02-14 10:42:58.247,3.607755,26.0,10,2025-02-14,2025-02-14 11:46:28 UTC,2025-02-14 12:02:28 UTC


In [5]:
# =====================================================
# 04) TIPOS Y LIMPIEZA
# =====================================================

df["fecha_creacion_fv"] = pd.to_datetime(
    df["fecha_creacion_fv"],
    errors="coerce",
    utc=True
)

for col in ["KM_REAL", "TIEMPO_ENTREGA_RETORNO"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["fecha_creacion_fv", "local"]).copy()

# Convertir a Lima si aplica
if USE_LIMA_TZ:
    df["fecha_creacion_fv"] = df["fecha_creacion_fv"].dt.tz_convert(LIMA_TZ)

# Bucket horario
df["ts_hour"] = df["fecha_creacion_fv"].dt.floor("h")
df["hora"] = df["ts_hour"].dt.hour

print("Shape luego de limpieza:", df.shape)
print("Rango real observado:", df["ts_hour"].min(), "->", df["ts_hour"].max())
print("Locales:", df["local"].nunique())


Shape luego de limpieza: (9504, 13)
Rango real observado: 2025-02-13 19:00:00-05:00 -> 2025-05-04 18:00:00-05:00
Locales: 41


In [6]:
# =====================================================
# 05) AGREGACIÓN BASE OBSERVADA (local-hora)
# =====================================================

base_observada = (
    df.groupby(["local", "ts_hour"], as_index=False)
      .agg(
          pedidos=("order_id", "count"),
          km_mean=("KM_REAL", "mean"),
          t_ret_mean=("TIEMPO_ENTREGA_RETORNO", "mean"),
          t_ret_p75=("TIEMPO_ENTREGA_RETORNO",
                     lambda x: np.nanpercentile(x, 75) if np.isfinite(x).any() else np.nan)
      )
)

base_observada["origen_dato"] = "real"

print("Shape base observada:", base_observada.shape)
display(base_observada.head())


Shape base observada: (1541, 7)


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,origen_dato
0,Caravana - San Borja,2025-02-14 07:00:00-05:00,1,3.714496,24.0,24.0,real
1,Caravana - San Borja,2025-04-06 07:00:00-05:00,1,1.060328,32.0,32.0,real
2,Caravana - San Borja,2025-04-20 08:00:00-05:00,1,3.390560,30.0,30.0,real
3,Caravana - San Borja,2025-04-27 07:00:00-05:00,1,0.953453,34.0,34.0,real
4,Caravana - San Borja,2025-04-27 08:00:00-05:00,1,0.647603,10.0,10.0,real


In [7]:
# =====================================================
# 06) GRILLA CALENDARIO POR LOCAL
#     Importante:
#     - Antes se llenaba pedidos=0 cuando faltaba una hora.
#     - En este caso, como el dataset es una muestra parcial,
#       una hora faltante NO significa necesariamente cero demanda.
#     - Por eso, se marca como faltante y luego CTGAN puede sintetizarla.
# =====================================================

def build_calendar_grid_per_local(base_df: pd.DataFrame) -> pd.DataFrame:
    out = []

    for loc, b in base_df.groupby("local", sort=False):
        min_ts = b["ts_hour"].min().floor("D")
        max_ts = b["ts_hour"].max().ceil("D") - pd.Timedelta(hours=1)

        hours = pd.date_range(
            start=min_ts,
            end=max_ts,
            freq=FREQ,
            tz=min_ts.tz
        )

        idx = pd.MultiIndex.from_product(
            [[loc], hours],
            names=["local", "ts_hour"]
        )

        out.append(idx.to_frame(index=False))

    return pd.concat(out, ignore_index=True)


grilla = build_calendar_grid_per_local(base_observada)

base_keys = base_observada[["local", "ts_hour"]].drop_duplicates()

grilla = grilla.merge(
    base_keys.assign(existe_real=1),
    on=["local", "ts_hour"],
    how="left"
)

grilla["existe_real"] = grilla["existe_real"].fillna(0).astype(int)

print("Total combinaciones local-hora en grilla:", len(grilla))
print("Combinaciones reales observadas:", grilla["existe_real"].sum())
print("Combinaciones faltantes:", (grilla["existe_real"] == 0).sum())

display(grilla.head())


Total combinaciones local-hora en grilla: 44592
Combinaciones reales observadas: 1541
Combinaciones faltantes: 43051


,local,ts_hour,existe_real
0,Caravana - San Borja,2025-02-14 00:00:00-05:00,0
1,Caravana - San Borja,2025-02-14 01:00:00-05:00,0
2,Caravana - San Borja,2025-02-14 02:00:00-05:00,0
3,Caravana - San Borja,2025-02-14 03:00:00-05:00,0
4,Caravana - San Borja,2025-02-14 04:00:00-05:00,0


In [8]:
# =====================================================
# 07) VARIABLES CALENDARIO Y FILTRO DE HORAS OPERATIVAS
# =====================================================

grilla["hora"] = grilla["ts_hour"].dt.hour
grilla["dow"] = grilla["ts_hour"].dt.dayofweek
grilla["month"] = grilla["ts_hour"].dt.month
grilla["is_weekend"] = (grilla["dow"] >= 5).astype(int)

base_observada["hora"] = base_observada["ts_hour"].dt.hour
base_observada["dow"] = base_observada["ts_hour"].dt.dayofweek
base_observada["month"] = base_observada["ts_hour"].dt.month
base_observada["is_weekend"] = (base_observada["dow"] >= 5).astype(int)

if FILTER_OPERATING_HOURS:
    pos_rate = (
        base_observada.assign(pos=(base_observada["pedidos"] > 0).astype(int))
                      .groupby(["local", "hora"])["pos"]
                      .mean()
                      .reset_index()
                      .rename(columns={"pos": "pos_rate"})
    )

    grilla = grilla.merge(pos_rate, on=["local", "hora"], how="left")
    grilla["pos_rate"] = grilla["pos_rate"].fillna(0)

    grilla = grilla[grilla["pos_rate"] >= OPER_HOUR_MIN_POS_RATE].copy()
    grilla.drop(columns=["pos_rate"], inplace=True)

    horas_operativas = grilla[["local", "hora"]].drop_duplicates()

    base_observada = base_observada.merge(
        horas_operativas.assign(es_hora_operativa=1),
        on=["local", "hora"],
        how="left"
    )

    base_observada = base_observada[
        base_observada["es_hora_operativa"].fillna(0).eq(1)
    ].copy()

    base_observada.drop(columns=["es_hora_operativa"], inplace=True)

print("Grilla luego de filtro operativo:", grilla.shape)
print("Base observada luego de filtro operativo:", base_observada.shape)


Grilla luego de filtro operativo: (21599, 7)
Base observada luego de filtro operativo: (1541, 11)


In [9]:
# =====================================================
# 08) GENERACIÓN SINTÉTICA CON CTGAN
# =====================================================

def preparar_datos_ctgan(base_real: pd.DataFrame) -> pd.DataFrame:
    train_ctgan = base_real[CTGAN_COLUMNS].copy()

    # Imputación robusta para variables operativas
    for col in ["km_mean", "t_ret_mean", "t_ret_p75"]:
        train_ctgan[col] = train_ctgan[col].fillna(train_ctgan[col].median())

    # SDV trabaja mejor si las categóricas están claramente definidas
    for col in CONDITION_COLUMNS:
        train_ctgan[col] = train_ctgan[col].astype(str)

    # Asegurar tipos numéricos
    for col in ["pedidos", "km_mean", "t_ret_mean", "t_ret_p75"]:
        train_ctgan[col] = pd.to_numeric(train_ctgan[col], errors="coerce")

    train_ctgan = train_ctgan.dropna().copy()
    train_ctgan["pedidos"] = train_ctgan["pedidos"].round().astype(int)

    return train_ctgan


if USE_SYNTHETIC_DATA:
    train_ctgan = preparar_datos_ctgan(base_observada)

    print("Shape train CTGAN:", train_ctgan.shape)
    display(train_ctgan.head())

    metadata = Metadata.detect_from_dataframe(
        data=train_ctgan,
        table_name="demanda_local_hora"
    )

    for col in CONDITION_COLUMNS:
        metadata.update_column(
            table_name="demanda_local_hora",
            column_name=col,
            sdtype="categorical"
        )

    for col in ["pedidos", "km_mean", "t_ret_mean", "t_ret_p75"]:
        metadata.update_column(
            table_name="demanda_local_hora",
            column_name=col,
            sdtype="numerical"
        )

    synthesizer = CTGANSynthesizer(
        metadata,
        epochs=CTGAN_EPOCHS,
        enforce_rounding=True,
        enforce_min_max_values=True,
        verbose=True
    )

    synthesizer.fit(train_ctgan)

    print("✅ CTGAN entrenado")
else:
    train_ctgan = None
    synthesizer = None
    print("⚠️ Generación sintética desactivada")


Shape train CTGAN: (1541, 9)


,local,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend
0,Caravana - San Borja,1,3.714496,24.0,24.0,7,4,2,0
1,Caravana - San Borja,1,1.060328,32.0,32.0,7,6,4,1
2,Caravana - San Borja,1,3.390560,30.0,30.0,8,6,4,1
3,Caravana - San Borja,1,0.953453,34.0,34.0,7,6,4,1
4,Caravana - San Borja,1,0.647603,10.0,10.0,8,6,4,1


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-00.63) | Discrim. (-00.01): 100%|██████████| 300/300 [01:37<00:00,  3.07it/s]

✅ CTGAN entrenado


In [10]:
# =====================================================
# 09) GENERAR DATA SINTÉTICA PARA COMBINACIONES FALTANTES
# =====================================================

MAX_SYNTHETIC_ROWS = 3000

if USE_SYNTHETIC_DATA:
    # Tomar combinaciones faltantes de la grilla
    faltantes = grilla[grilla["existe_real"] == 0].copy()

    # Limitar cantidad de filas sintéticas para evitar tiempos excesivos
    if MAX_SYNTHETIC_ROWS is not None:
        faltantes = faltantes.head(MAX_SYNTHETIC_ROWS).copy()

    n_synth = len(faltantes)

    print("Filas faltantes candidatas a sintetizar:", len(grilla[grilla["existe_real"] == 0]))
    print("Filas faltantes a sintetizar:", n_synth)

    if n_synth > 0:
        # Generación sintética rápida sin condiciones fila por fila
        synthetic_data = synthesizer.sample(num_rows=n_synth)

        # Asignar calendario real desde la grilla faltante
        # Esto permite que la data sintética cubra fechas/horas/locales faltantes
        synthetic_data["local"] = faltantes["local"].astype(str).values
        synthetic_data["ts_hour"] = faltantes["ts_hour"].values

        synthetic_data["hora"] = faltantes["hora"].astype(int).values
        synthetic_data["dow"] = faltantes["dow"].astype(int).values
        synthetic_data["month"] = faltantes["month"].astype(int).values
        synthetic_data["is_weekend"] = faltantes["is_weekend"].astype(int).values

        # Marcar origen del dato
        synthetic_data["origen_dato"] = "sintetico_ctgan"

        print("Shape data sintética:", synthetic_data.shape)
        display(synthetic_data.head())

    else:
        print("No existen combinaciones faltantes para sintetizar.")
        synthetic_data = pd.DataFrame()

else:
    synthetic_data = pd.DataFrame()


Filas faltantes candidatas a sintetizar: 20058
Filas faltantes a sintetizar: 3000
Shape data sintética: (3000, 11)


,local,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,ts_hour,origen_dato
0,Caravana - San Borja,3,2.775570,38.565627,27.0,6,4,2,0,2025-02-14 11:00:00,sintetico_ctgan
1,Caravana - San Borja,1,3.020673,58.633301,34.6,8,4,2,0,2025-02-14 13:00:00,sintetico_ctgan
2,Caravana - San Borja,5,0.772974,30.725571,50.1,9,4,2,0,2025-02-14 14:00:00,sintetico_ctgan
3,Caravana - San Borja,8,1.008546,23.083248,52.9,10,4,2,0,2025-02-14 15:00:00,sintetico_ctgan
4,Caravana - San Borja,16,2.054893,35.239234,41.2,6,5,2,1,2025-02-15 11:00:00,sintetico_ctgan


In [11]:
# =====================================================
# 10) POSTPROCESO Y REGLAS DE NEGOCIO SOBRE DATA SINTÉTICA
# =====================================================

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    # Corrección de tipos
    for col in CONDITION_COLUMNS:
        if col != "local":
            synthetic_data[col] = pd.to_numeric(
                synthetic_data[col],
                errors="coerce"
            ).round().astype("Int64")

    synthetic_data["local"] = synthetic_data["local"].astype(str)

    # Reglas de rango
    synthetic_data["pedidos"] = (
        pd.to_numeric(synthetic_data["pedidos"], errors="coerce")
        .fillna(0)
        .round()
        .clip(lower=0)
        .astype(int)
    )

    for col in ["km_mean", "t_ret_mean", "t_ret_p75"]:
        synthetic_data[col] = (
            pd.to_numeric(synthetic_data[col], errors="coerce")
            .fillna(train_ctgan[col].median())
            .clip(lower=0)
        )

    synthetic_data["hora"] = synthetic_data["hora"].clip(0, 23).astype(int)
    synthetic_data["dow"] = synthetic_data["dow"].clip(0, 6).astype(int)
    synthetic_data["month"] = synthetic_data["month"].clip(1, 12).astype(int)
    synthetic_data["is_weekend"] = synthetic_data["is_weekend"].clip(0, 1).astype(int)

    # Orden de columnas
    synthetic_data = synthetic_data[
        [
            "local",
            "ts_hour",
            "pedidos",
            "km_mean",
            "t_ret_mean",
            "t_ret_p75",
            "hora",
            "dow",
            "month",
            "is_weekend",
            "origen_dato"
        ]
    ].copy()

    print("✅ Postproceso sintético completado")
    display(synthetic_data.head())
else:
    print("No hay data sintética para postprocesar")


✅ Postproceso sintético completado


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,origen_dato
0,Caravana - San Borja,2025-02-14 11:00:00,3,2.775570,38.565627,27.0,6,4,2,0,sintetico_ctgan
1,Caravana - San Borja,2025-02-14 13:00:00,1,3.020673,58.633301,34.6,8,4,2,0,sintetico_ctgan
2,Caravana - San Borja,2025-02-14 14:00:00,5,0.772974,30.725571,50.1,9,4,2,0,sintetico_ctgan
3,Caravana - San Borja,2025-02-14 15:00:00,8,1.008546,23.083248,52.9,10,4,2,0,sintetico_ctgan
4,Caravana - San Borja,2025-02-15 11:00:00,16,2.054893,35.239234,41.2,6,5,2,1,sintetico_ctgan


In [13]:
# =====================================================
# 11) UNIÓN REAL + SINTÉTICO
# =====================================================

cols_base = [
    "local",
    "ts_hour",
    "pedidos",
    "km_mean",
    "t_ret_mean",
    "t_ret_p75",
    "hora",
    "dow",
    "month",
    "is_weekend",
    "origen_dato"
]

base_real = base_observada[cols_base].copy()

# -----------------------------------------------------
# Normalizar ts_hour en base real
# -----------------------------------------------------
base_real["ts_hour"] = pd.to_datetime(
    base_real["ts_hour"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

# -----------------------------------------------------
# Normalizar ts_hour en data sintética
# -----------------------------------------------------
if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    synthetic_data = synthetic_data.copy()

    synthetic_data["ts_hour"] = pd.to_datetime(
        synthetic_data["ts_hour"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

    base_full = pd.concat(
        [base_real, synthetic_data[cols_base]],
        ignore_index=True
    )
else:
    base_full = base_real.copy()

# -----------------------------------------------------
# Normalización final de tipos
# -----------------------------------------------------
base_full["local"] = base_full["local"].astype(str)

for col in ["hora", "dow", "month", "is_weekend"]:
    base_full[col] = pd.to_numeric(base_full[col], errors="coerce").astype("Int64")

for col in ["pedidos", "km_mean", "t_ret_mean", "t_ret_p75"]:
    base_full[col] = pd.to_numeric(base_full[col], errors="coerce")

# pedidos debe ser entero no negativo
base_full["pedidos"] = (
    base_full["pedidos"]
    .round()
    .clip(lower=0)
    .astype("Int64")
)

# Eliminar filas con fecha inválida, si existieran
base_full = base_full.dropna(subset=["ts_hour"]).copy()

# Ordenar base consolidada
base_full = base_full.sort_values(["local", "ts_hour"]).reset_index(drop=True)

print("Shape base consolidada:", base_full.shape)
print(base_full["origen_dato"].value_counts())
print("\nTipo de ts_hour:", base_full["ts_hour"].dtype)

display(base_full.head())


Shape base consolidada: (4541, 11)
origen_dato
sintetico_ctgan    3000
real               1541
Name: count, dtype: int64

Tipo de ts_hour: datetime64[ns]


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,origen_dato
0,Caravana - San Borja,2025-02-14 11:00:00,3,2.775570,38.565627,27.0,6,4,2,0,sintetico_ctgan
1,Caravana - San Borja,2025-02-14 12:00:00,1,3.714496,24.000000,24.0,7,4,2,0,real
2,Caravana - San Borja,2025-02-14 13:00:00,1,3.020673,58.633301,34.6,8,4,2,0,sintetico_ctgan
3,Caravana - San Borja,2025-02-14 14:00:00,5,0.772974,30.725571,50.1,9,4,2,0,sintetico_ctgan
4,Caravana - San Borja,2025-02-14 15:00:00,8,1.008546,23.083248,52.9,10,4,2,0,sintetico_ctgan


In [14]:
# =====================================================
# 12) FEATURES TEMPORALES
# =====================================================

base_full["hora"] = base_full["ts_hour"].dt.hour
base_full["dow"] = base_full["ts_hour"].dt.dayofweek
base_full["month"] = base_full["ts_hour"].dt.month
base_full["is_weekend"] = (base_full["dow"] >= 5).astype(int)

# Variables cíclicas
base_full["hora_sin"] = np.sin(2 * np.pi * base_full["hora"] / 24)
base_full["hora_cos"] = np.cos(2 * np.pi * base_full["hora"] / 24)
base_full["dow_sin"] = np.sin(2 * np.pi * base_full["dow"] / 7)
base_full["dow_cos"] = np.cos(2 * np.pi * base_full["dow"] / 7)

base_full = base_full.sort_values(["local", "ts_hour"]).reset_index(drop=True)

print("✅ Features temporales recalculadas")
display(base_full.head())


✅ Features temporales recalculadas


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,origen_dato,hora_sin,hora_cos,dow_sin,dow_cos
0,Caravana - San Borja,2025-02-14 11:00:00,3,2.775570,38.565627,27.0,11,4,2,0,sintetico_ctgan,2.588190e-01,-0.965926,-0.433884,-0.900969
1,Caravana - San Borja,2025-02-14 12:00:00,1,3.714496,24.000000,24.0,12,4,2,0,real,1.224647e-16,-1.000000,-0.433884,-0.900969
2,Caravana - San Borja,2025-02-14 13:00:00,1,3.020673,58.633301,34.6,13,4,2,0,sintetico_ctgan,-2.588190e-01,-0.965926,-0.433884,-0.900969
3,Caravana - San Borja,2025-02-14 14:00:00,5,0.772974,30.725571,50.1,14,4,2,0,sintetico_ctgan,-5.000000e-01,-0.866025,-0.433884,-0.900969
4,Caravana - San Borja,2025-02-14 15:00:00,8,1.008546,23.083248,52.9,15,4,2,0,sintetico_ctgan,-7.071068e-01,-0.707107,-0.433884,-0.900969


In [15]:
# =====================================================
# 13) LAGS
# =====================================================

for lag in LAGS:
    base_full[f"pedidos_lag_{lag}h"] = (
        base_full.groupby("local")["pedidos"].shift(lag)
    )

print("✅ Lags calculados:", LAGS)


✅ Lags calculados: [1, 2, 3, 24, 168]


In [16]:
# =====================================================
# 14) ROLLING
# =====================================================

for w in ROLL_WINDOWS:
    shifted = base_full.groupby("local")["pedidos"].shift(1)

    base_full[f"pedidos_roll_mean_{w}h"] = (
        shifted.groupby(base_full["local"])
               .rolling(window=w, min_periods=max(2, w // 3))
               .mean()
               .reset_index(level=0, drop=True)
    )

    base_full[f"pedidos_roll_std_{w}h"] = (
        shifted.groupby(base_full["local"])
               .rolling(window=w, min_periods=max(2, w // 3))
               .std()
               .reset_index(level=0, drop=True)
    )

print("✅ Rolling calculados:", ROLL_WINDOWS)


✅ Rolling calculados: [6, 12, 24, 168]


In [17]:
# =====================================================
# 15) VALIDACIÓN BÁSICA DE DATA SINTÉTICA
# =====================================================

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    print("Distribución de pedidos reales observados:")
    display(base_real["pedidos"].describe())

    print("Distribución de pedidos sintéticos:")
    display(synthetic_data["pedidos"].describe())

    comparacion_hora = pd.DataFrame({
        "real": base_real.groupby("hora")["pedidos"].mean(),
        "sintetico": synthetic_data.groupby("hora")["pedidos"].mean()
    })

    print("Pedidos promedio por hora: real vs sintético")
    display(comparacion_hora)

    try:
        synthetic_eval = synthetic_data[CTGAN_COLUMNS].copy()
        for col in CONDITION_COLUMNS:
            synthetic_eval[col] = synthetic_eval[col].astype(str)

        quality_report = evaluate_quality(
            real_data=train_ctgan,
            synthetic_data=synthetic_eval,
            metadata=metadata
        )

        print("Score de calidad sintética:", quality_report.get_score())
    except Exception as e:
        print("No se pudo calcular evaluate_quality:", str(e))
else:
    print("No aplica validación sintética")


Distribución de pedidos reales observados:


,pedidos
count,1541.000000
mean,6.167424
std,8.198147
min,1.000000
25%,1.000000
50%,3.000000
75%,8.000000
max,81.000000


Distribución de pedidos sintéticos:


,pedidos
count,3000.000000
mean,4.750000
std,5.954101
min,1.000000
25%,1.000000
50%,3.000000
75%,6.000000
max,81.000000


Pedidos promedio por hora: real vs sintético


,real,sintetico
hora,,
0,1.250000,NaN
1,1.000000,NaN
2,1.181818,NaN
3,1.333333,NaN
4,1.166667,NaN
5,2.333333,NaN
6,3.545455,4.861702
7,11.281879,4.522500
8,14.434483,5.255639


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 106.14it/s]|
Column Shapes Score: 62.32%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 163.75it/s]|
Column Pair Trends Score: 28.25%

Overall Score (Average): 45.28%

Score de calidad sintética: 0.45281967036416876


In [18]:
# =====================================================
# 16) FILTRADO FINAL MODEL-READY
# =====================================================

required = [
    f"pedidos_lag_{min(LAGS)}h",
    "pedidos_lag_24h",
    "pedidos_roll_mean_24h"
]

df_model = base_full.dropna(subset=required).copy()

print("✅ Shape model-ready:", df_model.shape)
print("Rango fechas:", df_model["ts_hour"].min(), "->", df_model["ts_hour"].max())
print("Locales:", df_model["local"].nunique())

print("\nDistribución pedidos model-ready:")
display(df_model["pedidos"].describe())

print("\nOrigen de datos model-ready:")
display(df_model["origen_dato"].value_counts())


✅ Shape model-ready: (3878, 28)
Rango fechas: 2025-02-16 11:00:00 -> 2025-05-04 23:00:00
Locales: 25

Distribución pedidos model-ready:


,pedidos
count,3878.0
mean,5.1787
std,6.64287
min,1.0
25%,1.0
50%,3.0
75%,7.0
max,81.0



Origen de datos model-ready:


,count
origen_dato,
sintetico_ctgan,2871
real,1007


In [19]:
# =====================================================
# 17) GUARDAR
# =====================================================

# Dataset model-ready con solo registros reales
df_model_real = df_model[df_model["origen_dato"] == "real"].copy()

df_model_real.to_parquet(OUTPUT_FILE_REAL, index=False)

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    synthetic_data.to_parquet(OUTPUT_FILE_SYNTH, index=False)
    df_model.to_parquet(OUTPUT_FILE_AUGMENTED, index=False)

print("✅ Dataset model-ready real guardado en:")
print(OUTPUT_FILE_REAL)

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    print("\n✅ Dataset sintético guardado en:")
    print(OUTPUT_FILE_SYNTH)

    print("\n✅ Dataset model-ready real + sintético guardado en:")
    print(OUTPUT_FILE_AUGMENTED)


✅ Dataset model-ready real guardado en:
/content/drive/MyDrive/AML_Final_Project/dataset_model_real.parquet

✅ Dataset sintético guardado en:
/content/drive/MyDrive/AML_Final_Project/dataset_sintetico_ctgan.parquet

✅ Dataset model-ready real + sintético guardado en:
/content/drive/MyDrive/AML_Final_Project/dataset_model_real_mas_sintetico.parquet
